# Utilitários do Pipeline

Funções compartilhadas entre os notebooks do pipeline (`01_ingestao_landing` até
`05_reconciliacao`), importadas via `%run` em vez de duplicadas em cada notebook.

**Funções disponíveis:**
- `merge_ou_cria(df, nome_tabela, colunas_chave)`: grava um DataFrame via MERGE
  idempotente, criando a tabela na primeira execução.
- `registrar_execucao(notebook, data_referencia, modo_execucao, status, mensagem_erro, inicio, fim)`:
  registra o resultado de uma execução na tabela `observability.pipeline_runs`.

Este notebook não deve ser executado sozinho como pipeline — ele existe para ser
importado por outros notebooks via `%run ../setup/01_utilitarios_pipeline`.

In [0]:
# imports
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import uuid

In [0]:
# funcao utilitaria de merge idempotente
def merge_ou_cria(df, nome_tabela, colunas_chave):
    condicao = " AND ".join([f"destino.{c} = origem.{c}" for c in colunas_chave])
    if spark.catalog.tableExists(nome_tabela):
        tabela_delta = DeltaTable.forName(spark, nome_tabela)
        (tabela_delta.alias("destino")
            .merge(df.alias("origem"), condicao)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"MERGE executado em {nome_tabela}")
    else:
        df.write.format("delta").saveAsTable(nome_tabela)
        print(f"Tabela {nome_tabela} criada pela primeira vez")

In [0]:
# funcao utilitaria de registro de execucao (observabilidade)
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType
from datetime import datetime as dt

schema_pipeline_runs = StructType([
    StructField("execucao_id", StringType(), True),
    StructField("notebook", StringType(), True),
    StructField("data_referencia", DateType(), True),
    StructField("modo_execucao", StringType(), True),
    StructField("status", StringType(), True),
    StructField("mensagem_erro", StringType(), True),
    StructField("inicio", TimestampType(), True),
    StructField("fim", TimestampType(), True),
])

def registrar_execucao(notebook, data_referencia, modo_execucao, status, inicio, fim, mensagem_erro=None):
    # converte data_referencia para objeto date, caso venha como string
    if isinstance(data_referencia, str):
        data_referencia = dt.strptime(data_referencia, "%Y-%m-%d").date()

    registro = spark.createDataFrame([{
        "execucao_id": str(uuid.uuid4()),
        "notebook": notebook,
        "data_referencia": data_referencia,
        "modo_execucao": modo_execucao,
        "status": status,
        "mensagem_erro": mensagem_erro,
        "inicio": inicio,
        "fim": fim,
    }], schema=schema_pipeline_runs)

    registro = registro.withColumn(
        "duracao_segundos",
        F.round(F.col("fim").cast("double") - F.col("inicio").cast("double"), 2)
    )

    (registro.write
        .format("delta")
        .mode("append")
        .saveAsTable("poc_b3_modernizacao.observability.pipeline_runs")
    )

    print(f"Execucao registrada: {notebook} | status={status}")